# Test Example 3: 4D Calibration with Clearer Products

This example features minimal aliasing and distinct nonlinear transformations:
- y0 = x0 * x1^2  (linear x0 coupled to quadratic x1)
- y1 = x1^3       (pure cubic)
- y2 = x2 * exp(x3)  (product of linear and exponential)
- y3 = x3^2       (pure quadratic)

Benefits:
- Each parameter appears in specific observables
- Nonlinearities are strong and distinct
- Varying noise scales highlight constraint differences
- Clean invertibility for testing

In [ ]:
%cd /Users/lucas/repositories/degeneracy_distillery/

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from pyoperon.sklearn import SymbolicRegressor
import multiprocessing
import csv
from sklearn.metrics import r2_score
import string
import sys,os
import sympy
import scipy

try:
    import esr.generation.generator
except ImportError:
    print("Warning: ESR not installed. Install with: git clone https://github.com/DeaglanBartlett/ESR.git && pip install -e ESR")

import jax
import jax.numpy as jnp
import jax.random as jr

from tqdm import tqdm as tq

sys.path.insert(0, '../degeneracy_distillery')

## Configuration

In [ ]:
n_params = 4
n_d = 200  # number of data points per sample
n_samples = 4000  # training samples
n_test = 1000  # test samples

## Transformation Functions

Forward: y = f(x)
- y0 = x0 * x1^2
- y1 = x1^3
- y2 = x2 * exp(x3)
- y3 = x3^2

Inverse: x = f^{-1}(y)

In [ ]:
def mu_y_from_x(x):
    """Product terms with clear separation - minimal aliasing"""
    y0 = x[0] * x[1]**2         # x0 linearly coupled to x1^2
    y1 = x[1]**3                # pure cubic
    y2 = x[2] * jnp.exp(x[3])   # product of linear and exponential
    y3 = x[3]**2                # pure quadratic
    return jnp.array([y0, y1, y2, y3])


def mu_x_from_y(y):
    """Inverse transformation - solve backwards"""
    # Work from independent equations first
    x1 = y[1]**(1/3)        # from y1 = x1^3
    x0 = y[0] / x1**2       # from y0 = x0 * x1^2
    x3 = jnp.sqrt(y[3])     # from y3 = x3^2
    x2 = y[2] / jnp.exp(x3) # from y2 = x2 * exp(x3)
    return jnp.array([x0, x1, x2, x3])


# Quick test of invertibility
test_x = jnp.array([0.5, 0.6, 0.4, 0.3])
test_y = mu_y_from_x(test_x)
test_x_reconstructed = mu_x_from_y(test_y)
print("Original x:", test_x)
print("Reconstructed x:", test_x_reconstructed)
print("Max error:", jnp.max(jnp.abs(test_x - test_x_reconstructed)))

## Noise Model

Varying noise scales to highlight which transforms are well-constrained:
- y0: σ = 0.05 (well constrained product)
- y1: σ = 0.02 (very well constrained cubic)
- y2: σ = 0.20 (loosely constrained product with exp)
- y3: σ = 0.08 (moderately constrained quadratic)

In [ ]:
Sigma = jnp.diag(jnp.array([0.05, 0.02, 0.20, 0.08]))

print("Noise covariance matrix:")
print(Sigma)

## Simulator

Simulates data in observable space Y, then transforms to parameter space X

In [ ]:
def simulator(key, θ, nd=n_d):
    """
    Simulate data: given parameters θ in Y space, transform to X and add noise
    
    Parameters:
    -----------
    key : JAX PRNG key
    θ : array, shape (n_params,)
        Parameters in Y space
    nd : int
        Number of data points to simulate
    
    Returns:
    --------
    x_samples : array, shape (nd * n_params,)
        Flattened simulated data in X space
    """
    # Transform Y -> X
    x = mu_x_from_y(θ)
    
    # Add Gaussian noise
    cov = Sigma
    return jr.multivariate_normal(key, mean=x, cov=cov, shape=(nd,)).reshape(-1)

## Generate Training and Test Data

Sample parameters θ uniformly in [0.2, 0.8] to keep transformations well-behaved

In [ ]:
print('Making data...')
key = jr.PRNGKey(0)
key1, key2 = jr.split(key)

# Sample parameters in [0.2, 0.8] for stable transformations
theta_ = jr.uniform(key1, shape=(n_samples, n_params), minval=0.2, maxval=0.8)

# Make test set
key1, key2 = jr.split(key1)
theta_test = jr.uniform(key1, shape=(n_test, n_params), minval=0.2, maxval=0.8)

# Create data
keys = jr.split(key, num=n_samples)
data_train = jax.vmap(simulator)(keys, theta_)

keys = jr.split(key2, num=n_test)
data_test = jax.vmap(simulator)(keys, theta_test)

print(f'Training data shape: {data_train.shape}')
print(f'Test data shape: {data_test.shape}')
print(f'Training parameters shape: {theta_.shape}')
print(f'Test parameters shape: {theta_test.shape}')

theta_train = theta_.copy()

## Visualize Parameter-Observable Relationships

Check how the transformations look in data space

In [ ]:
# Transform parameters to check ground truth y = f(x)
y_true = jax.vmap(mu_y_from_x)(theta_train[:500])
x_true = jax.vmap(mu_x_from_y)(theta_train[:500])

fig, axs = plt.subplots(n_params, n_params, figsize=(12, 12))

for i in range(n_params):
    for j in range(n_params):
        axs[i, j].scatter(x_true[:, i], y_true[:, j], alpha=0.5, s=10)
        axs[i, j].set_xlabel(f"$x_{i}$")
        axs[i, j].set_ylabel(f"$y_{j}$")
        axs[i, j].grid(alpha=0.3)

plt.suptitle("Ground Truth Transformations: y = f(x)")
plt.tight_layout()
plt.show()

print("\nExpected relationships:")
print("- y0 vs x0: linear")
print("- y0 vs x1: quadratic")
print("- y1 vs x1: cubic")
print("- y2 vs x2: linear")
print("- y2 vs x3: exponential")
print("- y3 vs x3: quadratic")

## Compute Fisher Matrices and Flatten

In [ ]:
from preprocessing_utils import (
    flatten_with_numerical_jacobian,
    batch_flatten_fisher
)

print('Computing Fisher matrices and flattening...')

# Step 1: Compute Jacobians dy/dx for all training samples
# The Jacobian is d(mu_y_from_x)/dx evaluated at each theta (in Y space)
def compute_jacobian_at_theta(theta):
    """Compute dy/dx at a single theta point"""
    # Transform theta (Y-space) to x (X-space)
    x = mu_x_from_y(theta)
    # Compute Jacobian of y = f(x) w.r.t. x
    J = jax.jacrev(mu_y_from_x)(x)
    return J

# Compute Jacobians for all training samples
print("Computing Jacobians...")
Js = jax.vmap(compute_jacobian_at_theta)(theta_train)
print(f'Jacobians shape: {Js.shape}')  # Should be (n_samples, n_params, n_params)

# Step 2: Compute Fisher information matrices
# Fisher = n_d * J^T @ Sigma^{-1} @ J
print("Computing Fisher matrices...")
Sigma_inv = jnp.linalg.inv(Sigma)

def compute_fisher(J):
    """Compute Fisher matrix from Jacobian"""
    return n_d * J.T @ Sigma_inv @ J

Fs = jax.vmap(compute_fisher)(Js)
print(f'Fisher matrices shape: {Fs.shape}')  # Should be (n_samples, n_params, n_params)

# Step 3: Flatten using the canonical function
print("Flattening Fisher matrices...")
Qs = batch_flatten_fisher(Js, Fs)
print(f'Flattened Fisher matrices shape: {Qs.shape}')  # Should be (n_samples, n_params, n_params)

# Check flattening quality: Q should be closer to identity than F
print("\nFlattening quality check:")
print(f"Mean Fisher eigenvalues: {jnp.linalg.eigvalsh(Fs.mean(0))}")
print(f"Mean flattened eigenvalues: {jnp.linalg.eigvalsh(Qs.mean(0))}")
print(f"(Flattened should be closer to all 1.0)")

# Visualize mean Fisher vs mean flattened
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(Fs.mean(0), cmap='RdBu_r', vmin=-2, vmax=2)
axes[0].set_title('Mean Fisher Matrix F')
axes[0].set_xlabel('Parameter j')
axes[0].set_ylabel('Parameter i')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(Qs.mean(0), cmap='RdBu_r', vmin=-2, vmax=2)
axes[1].set_title('Mean Flattened Matrix Q')
axes[1].set_xlabel('Parameter j')
axes[1].set_ylabel('Parameter i')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

## Analyze Flattened Coordinates

The flattened coordinates should reveal the structure of the transformations

In [ ]:
# Check a few sample Jacobians to see the structure
print("\nSample Jacobian (dy/dx) at random point:")
sample_idx = 0
J = Js[sample_idx]
print(f"Parameters (y-space): {theta_train[sample_idx]}")
print(f"\nJacobian dy/dx:")
print(J)
print("\nExpected structure:")
print("Row 0 (dy0): [x1^2, 2*x0*x1, 0, 0]")
print("Row 1 (dy1): [0, 3*x1^2, 0, 0]")
print("Row 2 (dy2): [0, 0, exp(x3), x2*exp(x3)]")
print("Row 3 (dy3): [0, 0, 0, 2*x3]")

# Verify with analytical derivatives
x_val = mu_x_from_y(theta_train[sample_idx])
print(f"\nX-space values: {x_val}")
print(f"\nAnalytical Jacobian:")
J_analytical = jnp.array([
    [x_val[1]**2, 2*x_val[0]*x_val[1], 0, 0],
    [0, 3*x_val[1]**2, 0, 0],
    [0, 0, jnp.exp(x_val[3]), x_val[2]*jnp.exp(x_val[3])],
    [0, 0, 0, 2*x_val[3]]
])
print(J_analytical)
print(f"\nNumerical vs Analytical error: {jnp.max(jnp.abs(J - J_analytical))}")

## Save Data for Pipeline

Save in format compatible with your preprocessing pipeline

In [ ]:
# Prepare for saving - format depends on your pipeline expectations
save_path = "/Users/lucas/repositories/degeneracy_distillery/data/test_ex3_4d_calibration.npz"

# You may need to adjust this based on your pipeline's expected format
# Check what format test_ex1 or test_ex2 use
print(f"Data ready to save to: {save_path}")
print("\nNote: Adjust save format based on your pipeline requirements")

## Notes on This Calibration Example

### Why This Should Work Better:

1. **No Additive Degeneracies**: Unlike your original (x0 + x1), each parameter affects observables independently or through clear products

2. **Strong Nonlinearities**: 
   - Cubic (x1^3) is harder to miss than quadratic
   - Exponential (exp(x3)) creates strong gradients
   - Products make parameter coupling obvious

3. **Varying Constraints**: Different noise levels simulate realistic scenarios where some observables are better measured

4. **Clean Structure**:
   - x1 appears in y0 and y1 (coupling)
   - x3 appears in y2 and y3 (coupling)
   - x0 and x2 appear once each (reference parameters)

### What to Look For:

- After flattening, Fisher matrices should show which parameters are constrained
- Jacobian dy/dx should clearly show the block structure
- Symbolic regression should recover expressions like `x1**3`, `x0*x1**2`, `exp(x3)`

### Next Steps:

1. Run through your preprocessing pipeline
2. Check if PCA/rotation reveals the structure
3. Apply symbolic regression to flattened coordinates
4. If this works cleanly, gradually add complexity (more aliasing, identity terms, etc.)

---

## Key Improvements Over Original 6D Example

### 1. No Additive Aliasing
**Original Problem:** `y0 = (x0 + x1) * (a * x2)^2`
- x0 and x1 are completely degenerate - any combination that sums to the same value gives identical y0
- Fisher matrix cannot distinguish between them

**This Example:** `y0 = x0 * x1^2`
- x0 and x1 have distinct functional roles
- Clear coupling through product, not addition

### 2. Stronger Nonlinearities
**Original:**
- Quadratic: x2^2
- Exponential: exp(0.5 * x3)
- Three identity mappings: x1, x4, x5

**This Example:**
- Cubic: x1^3 (harder to confuse with other polynomials)
- Full exponential: exp(x3) (stronger gradient than exp(0.5*x3))
- Quadratic: x3^2
- Product with exponential: x2 * exp(x3)
- NO identity mappings

### 3. Clearer Jacobian Structure

The Jacobian dy/dx has a clear block diagonal pattern:

```
[ x1²      2x0*x1    0         0        ]
[ 0        3x1²      0         0        ]
[ 0        0         exp(x3)   x2*exp(x3)]
[ 0        0         0         2x3      ]
```

**What this reveals:**
- **Block 1 (rows 0-1):** x0 and x1 couple through y0 and y1
- **Block 2 (rows 2-3):** x2 and x3 couple through y2 and y3
- The blocks are independent (zeros in off-diagonal blocks)
- Each parameter has a distinct nonlinear signature

### 4. Varying Noise Scales

**Original:** All dimensions had σ = 0.125 (uniform)

**This Example:** 
- y0: σ = 0.05 (tight constraint on x0*x1² product)
- y1: σ = 0.02 (very tight on x1³ - should be easiest to recover)
- y2: σ = 0.20 (loose constraint on exp term)
- y3: σ = 0.08 (moderate constraint on x3²)

This mimics realistic scenarios where different observables have different measurement precision.

### 5. Better Parameter Ranges

**Original:** Uniform [0, 1] can cause issues near boundaries

**This Example:** Uniform [0.2, 0.8]
- Avoids near-zero values that make products unstable
- Keeps exp(x3) in reasonable range (exp(0.2) ≈ 1.22, exp(0.8) ≈ 2.23)
- Ensures all transformations are well-conditioned

### Expected Pipeline Results

**Fisher Matrix Structure:**
- Should show strong correlations between (x0, x1) and (x2, x3)
- Weak/no correlation between the two blocks
- Eigenvalue spectrum should reveal 2 strong and 2 moderate modes

**After Flattening:**
- Should break into ~2 independent subspaces
- Linear combinations should separate the coupled pairs

**Symbolic Regression Targets:**
- Should easily find: `x1**3`, `x1**2`, `exp(x3)`, `x3**2`
- Products should emerge as: `x0*x1**2`, `x2*exp(x3)`

### Comparison to Your Original 6D Example

| Aspect | Original 6D | This 4D Calibration |
|--------|-------------|---------------------|
| Degeneracies | x0+x1 aliased | Each xi distinct |
| Identity maps | 3 (x1, x4, x5) | 0 |
| Strongest nonlinearity | x2^2, exp(0.5*x3) | x1^3, exp(x3) |
| Noise | Uniform 0.125 | Varying 0.02-0.20 |
| Jacobian structure | Mixed | Clean blocks |
| Invertibility | Ambiguous | Unique |

This example should make it much easier to verify that each stage of your pipeline (Fisher → flattening → rotation → SR) is working correctly!